In [ ]:
import os
import xarray as xr
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac_client
from scipy import ndimage as ndi
from distributed import LocalCluster
from pyproj import Transformer
import json
import geopandas as gpd
from shapely.geometry import Point, box
from shapely.ops import unary_union


In [ ]:
from dask_gateway import Gateway

"""

gate = Gateway("https://dask.user.eopf.eodc.eu", 
               proxy_address="tls://dask.user.eopf.eodc.eu:10000",
               auth="jupyterhub")

"""

# Dask Gateway automatically loads the correct configuration.
# So the code below is identical to the comment above
gate = Gateway()
cluster = gate.new_cluster()
cluster


LocalCluster(275e8bec, 'inproc://10.8.244.48/1124519/1', workers=1, threads=16, memory=98.23 GiB)

In [3]:
# Define AOI and convert to raster CRS
spatial_extent = {
    "west": 3.547150,
    "south": 50.865238,
    "east": 3.548602,
    "north": 50.866717,
}

bbox_4326 = [
    spatial_extent["west"],
    spatial_extent["south"],
    spatial_extent["east"],
    spatial_extent["north"],
]

# Convert AOI to UTM 33N (same as Sentinel-2 data)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32631", always_xy=True)
west_utm, south_utm = transformer.transform(
    spatial_extent["west"], spatial_extent["south"]
)
east_utm, north_utm = transformer.transform(
    spatial_extent["east"], spatial_extent["north"]
)

# Spatial slice parameters
x_slice = slice(west_utm, east_utm)
y_slice = slice(north_utm, south_utm)

In [4]:
%time
# Connect to the STAC catalog
catalog = pystac_client.Client.open("https://stac.core.eopf.eodc.eu")

# Search for Sentinel-2 L2A items within a specific bounding box and date range
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox_4326,
    datetime="2018-11-01/2020-02-01",
)

# Retrieve the list of matching items
items = list(search.items())
hrefs = [item.assets["product"].href for item in items]

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 4.77 μs


In [5]:
%time


def extract_time(ds):
    date_format = "%Y%m%dT%H%M%S"
    filename = ds.encoding["source"]
    date_str = os.path.basename(filename).split("_")[2]
    time = datetime.strptime(date_str, date_format)
    return ds.assign_coords(time=time)


datacube = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/measurements/reflectance/r10m",
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)

scl = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/conditions/mask/l2a_classification/r20m",  # Adjust if necessary
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)[["scl"]]


b11 = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/measurements/reflectance/r20m",  # Adjust if necessary
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)[["b11"]]


CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 5.48 μs


In [6]:
import geopandas as gpd

crops = {"maize":1200,"potatos":5100,"sugarbeet":8100,"barley":1500,"soy":4100}
crop_samples = {name:gpd.read_file("resources/"+ name + "_2019.geojson", driver='GeoJSON') for name,code in crops.items()}
crop_samples

/home/sdhinakaran/micromamba/envs/eopf-zarr/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
/home/sdhinakaran/micromamba/envs/eopf-zarr/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
/home/sdhinakaran/micromamba/envs/eopf-zarr/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
/home/sdhinakaran/micromamba/envs/eopf-zarr/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
/home/sdhinakaran/micromamba/envs/eopf-zarr/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


{'maize':            sampleID          area userConf validityTi split  LC    CT  IRR  \
 0  0000280864596FCC  15565.161377     None 2019-06-01   CAL  11  1200    0   
 1  00002808665D0242  57299.294968     None 2019-06-01   CAL  11  1200    0   
 2  0000280860B5D51A  50807.420483     None 2019-06-01   CAL  11  1200    0   
 3  000028085C34823C  12155.579527     None 2019-06-01   VAL  11  1200    0   
 4  000028084839A106   6970.325405     None 2019-06-01   VAL  11  1200    0   
 5  000028084B77A8E3   9613.034222     None 2019-06-01   VAL  11  1200    0   
 6  000028085DE6F57E   9571.395310     None 2019-06-01   CAL  11  1200    0   
 7  0000280860C6B5D1  22660.177588     None 2019-06-01  TEST  11  1200    0   
 8  000028085DF7229E   6235.259787     None 2019-06-01   CAL  11  1200    0   
 9  000028085D200D9F   8035.871979     None 2019-06-01   CAL  11  1200    0   
 
         location_id                                           geometry  
 0  0000280864596FCC  POLYGON ((4.78561 50.840

In [ ]:
# --- helpers ---
def _raster_bounds_polygon_32631(ds):
    """Shapely box of the datacube extent (expects x/y in EPSG:32631)."""
    xmin = float(ds.x.min())
    xmax = float(ds.x.max())
    ymin = float(ds.y.min())
    ymax = float(ds.y.max())
    return box(min(xmin, xmax), min(ymin, ymax), max(xmin, xmax), max(ymin, ymax))

def _random_point_in_geom(geom, rng=None, max_tries=10_000):
    """Rejection sample one point inside (multi)polygon."""
    if rng is None:
        rng = np.random.default_rng()
    minx, miny, maxx, maxy = geom.bounds
    for _ in range(max_tries):
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        p = Point(x, y)
        if geom.contains(p):
            return p
    raise RuntimeError("Failed to sample a point inside geometry within max_tries.")

# --- main ---
def point_sample_fields_with_cube_bounds(
    crop_samples: dict[str, gpd.GeoDataFrame],
    nr_iterations: int,
    datacube,                          # xr.Dataset with .rio.crs == EPSG:32631 and x/y coords
    buffer_m: float = 1.0,
    rng: np.random.Generator | None = None,
) -> dict[str, str]:
    """
    Returns {crop_name: GeoJSON FeatureCollection (EPSG:4326)}
    of 1 m buffered random points sampled ONLY where fields overlap the datacube extent.
    """
    # sanity: datacube must be in 32631
    if getattr(datacube, "rio", None) is None or datacube.rio.crs is None or datacube.rio.crs.to_epsg() != 32631:
        raise ValueError("datacube must carry CRS EPSG:32631 (e.g., ds.rio.write_crs('EPSG:32631')).")

    if rng is None:
        rng = np.random.default_rng()

    raster_poly = _raster_bounds_polygon_32631(datacube)  # in 32631

    # collect sampled points (in 32631)
    pts = {"name": [], "geometry": []}

    for name, gdf in crop_samples.items():
        # ensure input GDF has CRS, project to 32631
        if gdf.crs is None:
            # your files are 4326; set if missing
            gdf = gdf.set_crs("EPSG:4326")
        if gdf.crs.to_epsg() != 32631:
            gdf = gdf.to_crs("EPSG:32631")

        for geom in gdf.geometry:
            inter = geom.intersection(raster_poly)
            if inter.is_empty:
                continue
            if inter.geom_type in ("MultiPolygon", "GeometryCollection"):
                inter = unary_union([gg for gg in getattr(inter, "geoms", []) if gg.area > 0])

            for _ in range(nr_iterations):
                p = _random_point_in_geom(inter, rng=rng)
                pts["name"].append(name)
                pts["geometry"].append(p)

    # points → 1 m buffers in 32631
    gdf_points = gpd.GeoDataFrame(pts, crs="EPSG:32631")
    gdf_buffers = gpd.GeoDataFrame(
        {"name": gdf_points["name"], "geometry": gdf_points.buffer(buffer_m)},
        crs="EPSG:32631"
    )

    # return GeoJSON strings per crop in EPSG:4326 (to match your existing pipeline)
    points_per_type: dict[str, str] = {}
    for crop_name in sorted(gdf_buffers["name"].unique()):
        crop_gdf = gdf_buffers[gdf_buffers["name"] == crop_name]
        points_per_type[crop_name] = crop_gdf.to_json()

    return points_per_type

# crop_samples read in 4326 from your files:
crops = {"maize":1200,"potatos":5100,"sugarbeet":8100,"barley":1500,"soy":4100}
crop_samples = {
    name: gpd.read_file(f"resources/{name}_2019.geojson", driver="GeoJSON")
    for name, _ in crops.items()
}

datacube = datacube.rio.write_crs("EPSG:32631")  # ensure CRS

points_per_type = point_sample_fields_with_cube_bounds(
    crop_samples=crop_samples,
    nr_iterations=30,
    datacube=datacube,   # the 32631 grid defines the valid sampling area
    buffer_m=1.0
)


In [ ]:
def circular_kernel(radius: int) -> np.ndarray:
    r = int(radius)
    y, x = np.ogrid[-r:r+1, -r:r+1]
    k = (x*x + y*y) <= (r*r)
    return k.astype(np.float32)

def _dilate_with_convolve(mask_2d: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    if mask_2d.dtype != np.float32:
        mask_2d = mask_2d.astype(np.float32, copy=False)
    conv = ndi.convolve(mask_2d, kernel, mode="constant", cval=0.0)
    return conv > 0.0

def _apply_dilation_block(scl_block: np.ndarray, k1: np.ndarray, k2: np.ndarray) -> np.ndarray:
    scl = scl_block.astype(np.int16, copy=False)
    mask1 = (scl != 2) & (scl != 4) & (scl != 5) & (scl != 6) & (scl != 7)
    mask2 = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)
    dil1 = _dilate_with_convolve(mask1, k1)
    dil2 = _dilate_with_convolve(mask2, k2)
    return dil1 | dil2

def mask_scl_dilation(ds: xr.Dataset, *, time_band_name="time", scl_band_name="scl") -> xr.Dataset:
    if scl_band_name not in ds:
        raise ValueError(f"{scl_band_name!r} not found in dataset variables: {list(ds.data_vars)}")
    if time_band_name not in ds.dims:
        raise ValueError(f"{time_band_name!r} not found in dataset dimensions: {list(ds.dims)}")

    kernel1 = circular_kernel(radius=8)
    kernel2 = circular_kernel(radius=100)

    scl = ds[scl_band_name]
    cand_xy = [d for d in ["y", "x"] if d in scl.dims]
    if len(cand_xy) != 2:
        cand_xy = list(scl.dims[-2:])
    ydim, xdim = cand_xy

    dilated = xr.apply_ufunc(
        _apply_dilation_block,
        scl,
        input_core_dims=[[ydim, xdim]],
        output_core_dims=[[ydim, xdim]],
        kwargs={"k1": kernel1.astype(np.float32), "k2": kernel2.astype(np.float32)},
        dask="parallelized",
        vectorize=True,
        output_dtypes=[bool],
        dask_gufunc_kwargs={"allow_rechunk": True},
    ).rename("dilated_mask")

    bands_to_mask = [v for v in ds.data_vars if v != scl.name]

    out = ds.copy()
    for var in bands_to_mask:
        da = out[var]
        if np.issubdtype(da.dtype, np.integer):
            da = da.astype(np.float32)
        out[var] = da.where(~dilated)

    out = out.drop_vars(scl.name)

    keep = xr.concat([out[v].notnull().any((ydim, xdim)) for v in bands_to_mask], dim="vars").any("vars")
    out = out.sel({time_band_name: keep})
    return out

def compute_ndvi(ds: xr.Dataset) -> xr.DataArray:
    nir = ds["b08"].astype("float32")
    red = ds["b04"].astype("float32")
    num = nir - red
    den = nir + red
    ndvi = xr.where(den != 0, num / den, np.nan)
    ndvi.name = "NDVI"
    return ndvi

def _axis_is_ascending(coord) -> bool:
    # robust to dask-backed coords
    first = float(coord[0])
    last  = float(coord[-1])
    return last >= first

def _slices_for_bbox(ds: xr.Dataset, bbox, pad: float = 0.0):
    """Return (x_slice, y_slice) for ds given bbox=(minx,miny,maxx,maxy) in ds CRS.
    Handles ascending/descending axes and optional padding in CRS units (meters)."""
    minx, miny, maxx, maxy = bbox
    if pad:
        minx -= pad; miny -= pad; maxx += pad; maxy += pad

    x_asc = _axis_is_ascending(ds.x)
    y_asc = _axis_is_ascending(ds.y)

    x_slice = slice(minx, maxx) if x_asc else slice(maxx, minx)
    # y in many satellite grids is descending; .sel must match axis direction
    y_slice = slice(miny, maxy) if y_asc else slice(maxy, miny)
    return x_slice, y_slice


# -------------------------
# Core function
# -------------------------
def process_samples_from_open_datasets(
    datacube: xr.Dataset,  # ["b04","b08"] @ 10 m
    scl: xr.Dataset,       # ["scl"] @ 20 m
    b11: xr.Dataset,       # ["b11"] @ 20 m
    points_per_type: dict[str, str],
    *,
    crs: str = "EPSG:32631",
) -> dict[str, list[xr.Dataset]]:
    """
    Use already-open xarray Datasets (lazy) and extract per-point samples.
    Returns { crop_type: [xr.Dataset, ...] } with NDVI computed and SCL-dilation applied.
    """

    # Sanity: have CRS for spatial clipping
    if getattr(datacube, "rio", None) is None or datacube.rio.crs is None:
        datacube = datacube.rio.write_crs(crs)
    if getattr(scl, "rio", None) is None or scl.rio.crs is None:
        scl = scl.rio.write_crs(crs)
    if getattr(b11, "rio", None) is None or b11.rio.crs is None:
        b11 = b11.rio.write_crs(crs)

    out: dict[str, list[xr.Dataset]] = {}

    for crop_type, geojson_str in points_per_type.items():
        gdf = gpd.GeoDataFrame.from_features(json.loads(geojson_str)["features"], crs="EPSG:32631")

        ds_list: list[xr.Dataset] = []
        for idx, row in gdf.iterrows():
            geom = row.geometry
            bbox = geom.bounds  # (minx, miny, maxx, maxy) in EPSG:32631

            # 1) fast windowed crop by bbox using .sel
            # (optional: add pad=20 to give a 20 m buffer)
            x_slice, y_slice = _slices_for_bbox(datacube, bbox, pad=10)
            print(x_slice, y_slice)

            dc_cut  = datacube.sel(x=x_slice, y=y_slice)
            scl_cut = scl.sel(x=x_slice, y=y_slice)
            b11_cut = b11.sel(x=x_slice, y=y_slice)

            # 2) resample 20 m → 10 m to align with datacube grid
            scl_resampled = scl_cut["scl"].interp_like(dc_cut, method="nearest")
            b11_resampled = b11_cut["b11"].interp_like(dc_cut, method="nearest")

            ds = xr.Dataset(
                data_vars={
                    "b04": dc_cut["b04"],
                    "b08": dc_cut["b08"],
                    "scl": scl_resampled,
                    "b11": b11_resampled,
                },
                coords=dc_cut.coords,
                attrs={"crop_type": crop_type, "feature_index": int(idx)},
            )

            # 3) (optional) refine within-window to the actual polygon mask
            # If you still want polygon-accurate masking, keep the box crop above
            # and then mask out pixels outside 'geom' with rasterio.rio.clip or a polygon mask.
            # If bbox-only is sufficient, skip this.

            ds_masked = mask_scl_dilation(ds, time_band_name="time", scl_band_name="scl")
            ndvi = compute_ndvi(ds_masked)
            ds_final = ds_masked.assign({"NDVI": ndvi})

            ds_list.append(ds_final.compute())


        out[crop_type] = ds_list

    return out

out = process_samples_from_open_datasets(
    datacube=datacube,  # your already-open 10 m dataset
    scl=scl,            # your already-open SCL@20 m
    b11=b11,            # your already-open B11@20 m
    points_per_type=points_per_type,
    crs="EPSG:32631"
)
out